# Sending requests through the gateway

A signed call from code: one the mandate allows, one it denies, then a narrower mandate that is issued, used and revoked.

Needs `make hack` first (keys, a `SCH01` mandate and the gateway on `:8082`). On `EXPIRED`, run `make gateway_issue`.

In [ ]:
import json
import os
import urllib.error
import urllib.request
from pathlib import Path

if Path.cwd().name == "examples":  # Jupyter starts in the notebook's folder; the paths below are from the repo root
    os.chdir("..")

from gateway.cli import DEFAULT_DENY, issue_mandate, revoke_mandate, sign_request
from gateway.lib.common import priv_from_pem

STATE = Path("gateway/state")
KEY = priv_from_pem((STATE / "agent.pem").read_bytes())
MANDATE = (STATE / "mandate.jwt").read_text().strip()


def call(tool: str, arguments: dict, mandate: str = MANDATE) -> dict:
    body = sign_request(KEY, server="pupil-data", tool=tool, arguments=arguments, mandate=mandate)
    req = urllib.request.Request(
        "http://localhost:8082/tools/call", data=json.dumps(body).encode(), headers={"content-type": "application/json"}
    )
    try:
        with urllib.request.urlopen(req) as r:  # noqa: S310
            return json.load(r)
    except urllib.error.HTTPError as e:  # a 403 still carries the decision
        return json.load(e)

## 1. A call the mandate allows

`search` at `SCH01` is in the starter mandate's scope: `ALLOW`, `IN_SCOPE`, and one row back.

In [ ]:
call("search", {"query": "P0004", "urn": "SCH01", "limit": 1})

## 2. A call it denies

Same tool, the school next door. The `urn` constraint only lists `SCH01`, so the gateway answers `403` with the reason and no `result`.

In [ ]:
call("search", {"query": "P0004", "urn": "SCH02", "limit": 1})

## 3. Issue a narrower mandate

A mandate can't be edited, so a change means a new one. This one is `search` only, `SCH02` only, for 10 minutes, for the same key. Keep `idx`: it is the mandate's place in the revocation list.

In [ ]:
narrow, idx = issue_mandate(
    STATE,
    KEY.public_key(),
    scope=[{"server": "pupil-data", "tools": ["search"], "constraints": {"urn": ["SCH02"]}}],
    deny=DEFAULT_DENY,
    ttl=600,
    purpose="Pupil-data lookups for school SCH02",
    principal="officer:jsmith",
    org="school:SCH02",
)
call("search", {"query": "P0004", "urn": "SCH02", "limit": 1}, mandate=narrow)

## 4. Revoke it

The gateway reads the status list on every request, so the same call is refused straight away with `REVOKED`. Your starter mandate is untouched.

In [ ]:
revoke_mandate(STATE, idx)
call("search", {"query": "P0004", "urn": "SCH02", "limit": 1}, mandate=narrow)

## Next

- For a model-driven agent, give the model `call` as its tool. The agent loop is yours to write.
- `GET localhost:8082/describe` lists the tools and the signing rules.
- To sign a claim of your own into a mandate (a clearance, a limit), pass `extra={...}` to `issue_mandate`: see [docs/new_mandate.md](../docs/new_mandate.md).